# DS102 V10 — seed ensemble of V6
Upload ds102_v7_bundle.zip. All fits on hosted Colab. No pretrained or outside data.
V6 user-reported Kaggle: 0.77269. V10 has not been trained or scored.


In [ ]:
import os, platform, zipfile, hashlib, json
from pathlib import Path
assert platform.system() == 'Linux' and os.environ.get('COLAB_RELEASE_TAG'), 'Hosted Colab only'
bundle = Path('/content/ds102_v7_bundle.zip')
if not bundle.exists():
    from google.colab import files
    uploaded = files.upload()
    assert bundle.exists(), 'Upload ds102_v7_bundle.zip'
    del uploaded
root = Path('/content/ds102_v7_bundle')
root.mkdir(exist_ok=True)
with zipfile.ZipFile(bundle) as z:
    for name in z.namelist():
        assert (root/name).resolve().is_relative_to(root.resolve()), 'Unsafe ZIP path'
    z.extractall(root)
manifest = json.loads((root/'bundle_manifest.json').read_text())
for name, expected in manifest.items():
    assert hashlib.sha256((root/name).read_bytes()).hexdigest() == expected, name
print('Bundle checked:', len(manifest), 'files. Hosted Colab CPU:', os.cpu_count())


In [ ]:
%pip -q install catboost==1.2.10 scikit-learn==1.7.2

In [ ]:
from pathlib import Path
root=Path('/content/ds102_v7_bundle')
root.mkdir(exist_ok=True)
sources={'experiment_v10.py': '"""Hosted-Colab-only seed ensemble of v6; no pretrained models or dev fitting.\n\nOne fixed candidate: average latent models with seeds 102,17,2026.\nKeep v6\'s .7/.3 mixture, weights, features and all other hyperparameters.\n"""\nimport gc, hashlib, importlib.metadata, json, os, platform, sys, time\nfrom pathlib import Path\nsys.path.insert(0, str(Path(__file__).resolve().parent/\'claude_review\'))\nimport numpy as np\nimport pandas as pd\nfrom sklearn.model_selection import StratifiedGroupKFold\nfrom train_submit import load_data, metrics\nfrom improved_features import prepare\nfrom latent_v6 import LatentFeatures, make_model\nfrom review_common import near_dup_groups, max_jaccard, class1_strata, macro\nfrom codex_checked_common import paired_cluster_bootstrap\n\nROOT=Path(__file__).resolve().parent\nOUT=ROOT/\'output_v10_experiment\'\nCONFIG=dict(seeds=[102,17,2026],svd_seed=\'same as model seed\',split_seed=102,\n    folds=3,alpha=.3,weights=[1.,1.2,1.2,.9],trees=800,depth=5,learning_rate=.06,\n    min_oof_delta=.001,min_positive_folds=2,min_bootstrap_q05=0.,\n    max_novel_fake_f1_drop=.01,max_dev_drop=.005)\n\n\ndef guard():\n    if platform.system()!=\'Linux\' or not os.environ.get(\'COLAB_RELEASE_TAG\'):\n        raise RuntimeError(\'All fitting must run on hosted Colab.\')\n\n\ndef checked(p,n):\n    p=np.asarray(p)\n    assert p.shape==(n,4) and np.isfinite(p).all()\n    assert (p>=0).all() and np.allclose(p.sum(1),1,atol=1e-5)\n    return p\n\n\ndef predict(base,latent):\n    return ((.7*base+.3*latent)*np.array(CONFIG[\'weights\'])).argmax(1)\n\n\ndef fit_predict(train,frames,seed):\n    guard()\n    f=LatentFeatures();f.svd.set_params(random_state=seed)\n    x=f.transform(train,fit=True)\n    model=make_model();model.set_params(random_seed=seed,thread_count=2)\n    model.fit(x,train.spam_label.to_numpy(),verbose=False)\n    assert np.array_equal(model.classes_,[0,1,2,3])\n    result=[checked(model.predict_proba(f.transform(df)),len(df)) for df in frames]\n    del f,x,model;gc.collect()\n    return result\n\n\ndef main():\n    guard();OUT.mkdir(exist_ok=True)\n    data=load_data(ROOT/\'data\')\n    train,dev,test=[prepare(data[k]) for k in [\'train\',\'dev\',\'test\']]\n    y=train.spam_label.to_numpy()\n    ref=np.load(ROOT/\'colab_reference.npz\',allow_pickle=False)\n    assert np.array_equal(ref[\'y\'],y)\n    for name,n in [(\'oof\',len(y)),(\'dev\',len(dev)),(\'test\',len(test))]:checked(ref[name],n)\n    paths=list(ROOT.glob(\'*.py\'))+list((ROOT/\'claude_review\').glob(\'*.py\'))+list((ROOT/\'data\').glob(\'*.csv\'))+[ROOT/\'colab_reference.npz\']\n    manifest=dict(config=CONFIG,versions={k:importlib.metadata.version(k) for k in [\'numpy\',\'scipy\',\'pandas\',\'scikit-learn\',\'catboost\']},\n        sha256={str(p.relative_to(ROOT)):hashlib.sha256(p.read_bytes()).hexdigest() for p in paths})\n    mid=hashlib.sha256(json.dumps(manifest,sort_keys=True).encode()).hexdigest()\n    mp=OUT/\'manifest.json\'\n    if mp.exists():assert json.loads(mp.read_text())==manifest,\'Manifest changed; use a fresh output directory.\'\n    mp.write_text(json.dumps(manifest,indent=2))\n    report=dict(config=CONFIG,baseline_user_reported_kaggle=.77269,\n        execution=\'hosted Google Colab CPU\',pretrained=False,fit_data=\'train only\',\n        caveat=\'Fixed candidate after many prior experiments; bootstrap is conditional and does not remove selection bias. Kaggle improvement unknown.\')\n    def status(stage,**kv):\n        report.update(stage=stage,**kv)\n        (OUT/\'report.json\').write_text(json.dumps(report,indent=2))\n        print(stage,kv,flush=True)\n    start=time.time()\n    oof=np.zeros((3,len(y),4));folds=np.full(len(y),-1)\n    split=StratifiedGroupKFold(n_splits=3,shuffle=True,random_state=102)\n    for fold,(tr,va) in enumerate(split.split(train,y,train.comment)):\n        assert np.all(ref[\'folds\'][va]==fold),\'Reference split mismatch\'\n        assert not set(train.comment.iloc[tr])&set(train.comment.iloc[va])\n        folds[va]=fold\n        for j,seed in enumerate(CONFIG[\'seeds\']):\n            status(\'cv_fit\',fold=fold+1,seed=seed)\n            ck=OUT/f\'fold{fold}_seed{seed}.npz\'\n            if ck.exists():\n                z=np.load(ck,allow_pickle=False)\n                assert z[\'manifest_id\'].item()==mid and np.array_equal(z[\'tr\'],tr) and np.array_equal(z[\'va\'],va)\n                p=checked(z[\'prediction\'],len(va))\n            else:\n                p=fit_predict(train.iloc[tr],[train.iloc[va].drop(columns=\'spam_label\')],seed)[0]\n                tmp=ck.with_suffix(\'.tmp.npz\')\n                np.savez_compressed(tmp,manifest_id=mid,tr=tr,va=va,prediction=p)\n                tmp.replace(ck)\n            oof[j,va]=p\n        status(\'fold_complete\',fold=fold+1)\n    old=predict(ref[\'oof\'],oof[0]);new=predict(ref[\'oof\'],oof.mean(0))\n    groups=near_dup_groups(train.comment.tolist(),thr=.7)\n    template=np.zeros(len(y),bool)\n    for fold in range(3):\n        va=np.flatnonzero(folds==fold);tr=np.flatnonzero((folds!=fold)&(y==1))\n        template[va]=max_jaccard(train.comment.iloc[va].tolist(),train.comment.iloc[tr].tolist())>=.6\n    delta=macro(y,new)-macro(y,old)\n    fd=[macro(y[folds==f],new[folds==f])-macro(y[folds==f],old[folds==f]) for f in range(3)]\n    novel_delta=class1_strata(y,new,template)[\'novel\'][\'f1\']-class1_strata(y,old,template)[\'novel\'][\'f1\']\n    boot=paired_cluster_bootstrap(y,old,new,groups,n=2000)\n    gates=dict(macro=delta>=.001,folds=sum(v>0 for v in fd)>=2,\n               novel=novel_delta>=-.01,bootstrap=boot[\'q05\']>0)\n    status(\'cv_complete\',baseline_oof=metrics(y,old),candidate_oof=metrics(y,new),\n        delta=delta,fold_deltas=fd,novel_fake_f1_delta=novel_delta,bootstrap=boot,gates=gates)\n    np.savez_compressed(OUT/\'oof.npz\',latent=oof,folds=folds,y=y)\n    if not all(gates.values()):\n        status(\'rejected_on_cv\',elapsed_seconds=time.time()-start);return\n    # Keep dev completely outside fitting; labels used only after full fits.\n    ext=[]\n    frames=[dev.drop(columns=\'spam_label\'),test]\n    for seed in CONFIG[\'seeds\']:\n        status(\'full_train_fit\',seed=seed)\n        ck=OUT/f\'full_seed{seed}.npz\'\n        if ck.exists():\n            z=np.load(ck,allow_pickle=False);assert z[\'manifest_id\'].item()==mid\n            result=[checked(z[\'dev\'],len(dev)),checked(z[\'test\'],len(test))]\n        else:\n            result=fit_predict(train,frames,seed)\n            tmp=ck.with_suffix(\'.tmp.npz\')\n            np.savez_compressed(tmp,manifest_id=mid,dev=result[0],test=result[1]);tmp.replace(ck)\n        ext.append(result)\n    dev_latent=np.stack([p[0] for p in ext])\n    test_latent=np.stack([p[1] for p in ext])\n    dp_old=predict(ref[\'dev\'],dev_latent[0]);dp=predict(ref[\'dev\'],dev_latent.mean(0))\n    report[\'dev\']=dict(baseline=metrics(dev.spam_label,dp_old),candidate=metrics(dev.spam_label,dp))\n    if macro(dev.spam_label,dp)<macro(dev.spam_label,dp_old)-.005:\n        status(\'rejected_on_dev\',elapsed_seconds=time.time()-start);return\n    labels=predict(ref[\'test\'],test_latent.mean(0))\n    sub=data[\'samplesubmission\'].copy()\n    sub[\'spam_label\']=sub.ID.astype(str).map(dict(zip(test.id.astype(str),labels))).astype(int)\n    assert len(sub)==len(test) and sub.ID.is_unique and sub.spam_label.isin(range(4)).all()\n    path=OUT/\'submission_v10_candidate.csv\';sub.to_csv(path,index=False)\n    pd.testing.assert_frame_equal(pd.read_csv(path),sub,check_dtype=False)\n    status(\'candidate_ready_not_submitted\',rows=len(sub),sha256=hashlib.sha256(path.read_bytes()).hexdigest(),elapsed_seconds=time.time()-start)\n\n\nif __name__==\'__main__\':main()\n', 'latent_v6.py': '"""Train-only latent TF-IDF review/product relations + numeric CatBoost.\n\nTruncatedSVD is fitted from scratch inside each training fold. No pretrained\nembeddings, language models or outside corpora are used.\n"""\nimport gc,json,time,traceback\nfrom pathlib import Path\nimport numpy as np\nimport pandas as pd\nfrom scipy import sparse\nfrom sklearn.decomposition import TruncatedSVD\nfrom sklearn.preprocessing import normalize\nfrom sklearn.model_selection import StratifiedGroupKFold\nfrom catboost import CatBoostClassifier\nfrom improved_features import FeaturesV2,dense_features,prepare\nfrom experiment_v6 import hosted_guard\nfrom selection_v5 import cross_weight\nfrom select_v2 import macro\nfrom train_submit import load_data,SEED\n\n\nclass LatentFeatures:\n    def __init__(self):\n        self.base=FeaturesV2()\n        self.svd=TruncatedSVD(n_components=64,n_iter=5,random_state=SEED)\n\n    def transform(self,df,fit=False):\n        if fit:\n            review=sparse.hstack([self.base.word.fit_transform(df.comment),\n                                  self.base.char.fit_transform(df.comment)],format=\'csr\')\n            review=normalize(review)\n            z=self.svd.fit_transform(review)\n        else:\n            review=sparse.hstack([self.base.word.transform(df.comment),\n                                  self.base.char.transform(df.comment)],format=\'csr\')\n            review=normalize(review)\n            z=self.svd.transform(review)\n        title=normalize(sparse.hstack([self.base.word.transform(df.product_name),\n                                      self.base.char.transform(df.product_name)],format=\'csr\'))\n        t=self.svd.transform(title)\n        cosine=np.asarray(review.multiply(title).sum(axis=1)).ravel()\n        x=np.hstack([z,t,z*t,np.abs(z-t),cosine[:,None],dense_features(df)])\n        x=pd.DataFrame(x.astype(np.float32),columns=[f\'f_{i}\' for i in range(x.shape[1])])\n        x[\'category\']=df.category.astype(str).to_numpy()\n        x[\'rating\']=df.rating.astype(str).to_numpy()\n        return x\n\n\ndef make_model():\n    return CatBoostClassifier(iterations=800,depth=5,learning_rate=.06,l2_leaf_reg=8,\n        loss_function=\'MultiClass\',auto_class_weights=\'SqrtBalanced\',\n        cat_features=[\'category\',\'rating\'],max_ctr_complexity=1,\n        random_seed=SEED,thread_count=2,allow_writing_files=False,verbose=False)\n\n\ndef main(data_dir=\'/content/ds102_data\',output_dir=\'/content/ds102_v6_latent\',reference_path=\'/content/colab_reference.npz\'):\n    hosted_guard()\n    out=Path(output_dir);out.mkdir(parents=True,exist_ok=True)\n    def status(stage,**values):\n        (out/\'status.json\').write_text(json.dumps(dict(stage=stage,time=time.time(),**values),indent=2))\n    try:\n        df=prepare(load_data(Path(data_dir))[\'train\']);y=df.spam_label.to_numpy()\n        ref=np.load(reference_path);assert np.array_equal(y,ref[\'y\'])\n        trees=[400,800];oof=np.zeros((2,len(y),4),dtype=np.float32)\n        folds=np.full(len(y),-1);results=[[],[]]\n        split=StratifiedGroupKFold(n_splits=3,shuffle=True,random_state=SEED)\n        for f,(tr,va) in enumerate(split.split(df,y,groups=df.comment)):\n            folds[va]=f;status(\'fit_svd\',fold=f+1,results=results)\n            features=LatentFeatures();xt=features.transform(df.iloc[tr],fit=True)\n            xv=features.transform(df.iloc[va].drop(columns=\'spam_label\'))\n            status(\'fit_catboost\',fold=f+1,results=results)\n            m=make_model();m.fit(xt,y[tr],verbose=False)\n            for i,n in enumerate(trees):\n                oof[i,va]=m.predict_proba(xv,ntree_end=n)\n                results[i].append(macro(y[va],oof[i,va].argmax(1)))\n            del features,m,xt,xv;gc.collect()\n            np.savez_compressed(out/\'partial.npz\',oof=oof,y=y,folds=folds)\n        assert np.array_equal(folds,ref[\'folds\'])\n        rows=[]\n        for i,n in enumerate(trees):\n            for alpha in [.15,.3,.5]:\n                p=(1-alpha)*ref[\'oof\']+alpha*oof[i]\n                row=cross_weight(p,y,folds)\n                raw=[macro(y[folds==f],p[folds==f].argmax(1)) for f in np.unique(folds)]\n                row[\'fold_macro_f1\']=[a+b for a,b in zip(raw,row[\'fold_deltas\'])] if row[\'use_weights\'] else raw\n                row.update(name=f\'latent_cat_{n}_{alpha}\',recipe={\'kind\':\'latent\',\'trees\':n,\'alpha\':alpha})\n                rows.append(row)\n        rows.sort(key=lambda r:r[\'selection_score\'],reverse=True)\n        np.savez_compressed(out/\'oof.npz\',oof=oof,y=y,folds=folds)\n        (out/\'selection.json\').write_text(json.dumps(rows,indent=2))\n        report={\'best\':rows[0],\'fold_scores\':results,\'pretrained\':False,\'fit_data\':\'train only\',\n                \'representation\':\'TF-IDF and 64D SVD fitted on each training fold; review/title relations\',\n                \'caveat\':\'Repeated OOF selection; weighting checks not fully nested CV.\'}\n        (out/\'cv_results.json\').write_text(json.dumps(report,indent=2));status(\'complete\',**report)\n        return report\n    except BaseException as exc:\n        status(\'error\',error=str(exc),traceback=traceback.format_exc());raise\n', 'experiment_v6.py': '"""Hosted Colab: template neighbours and probability-calibrated kernel SVM.\n\nNo pretrained models or outside data. All feature fitting is inside the same\ngrouped train folds as previous versions. Dev labels are not read for selection.\n"""\nimport gc,json,time,traceback,os,platform\nfrom pathlib import Path\nimport numpy as np\nfrom sklearn.preprocessing import normalize\nfrom sklearn.svm import SVC\nfrom sklearn.model_selection import StratifiedGroupKFold\nfrom sklearn.metrics.pairwise import rbf_kernel\nfrom improved_features import prepare,FeaturesV2\nfrom train_submit import load_data,metrics,SEED\nfrom select_v2 import macro\nfrom selection_v5 import cross_weight\n\n\ndef hosted_guard():\n    if platform.system()!=\'Linux\' or not os.environ.get(\'COLAB_RELEASE_TAG\'):\n        raise RuntimeError(\'Hosted Google Colab required; local training disabled.\')\n\n\ndef kernel(text_a,meta_a,text_b,meta_b):\n    k=(text_a@text_b.T).toarray()\n    k+=1;np.square(k,out=k)\n    k+=.5*rbf_kernel(meta_a,meta_b,gamma=.3)\n    k/=4.5\n    return np.ascontiguousarray(k,dtype=np.float64)\n\n\ndef neighbours(query,train,y,batch_size=256):\n    """Eight nearest TF-IDF reviews; all references belong to fitting data."""\n    counts=np.bincount(y,minlength=4)\n    class_weight=np.sqrt(len(y)/(4*counts))\n    probs=np.zeros((query.shape[0],4),dtype=np.float32)\n    maximum=np.zeros(query.shape[0],dtype=np.float32)\n    k=min(8,len(y))\n    for start in range(0,query.shape[0],batch_size):\n        sim=(query[start:start+batch_size]@train.T).toarray()\n        idx=np.argpartition(sim,-k,axis=1)[:,-k:]\n        values=np.take_along_axis(sim,idx,axis=1)\n        weights=np.maximum(values-.2,0)**4\n        labels=y[idx]\n        votes=np.stack([(weights*(labels==c)).sum(axis=1)*class_weight[c] for c in range(4)],axis=1)\n        votes+=1e-8\n        probs[start:start+len(sim)]=votes/votes.sum(axis=1,keepdims=True)\n        maximum[start:start+len(sim)]=values.max(axis=1)\n    return probs,maximum\n\n\ndef make_kernel_model(y):\n    counts=np.bincount(y,minlength=4)\n    cw={c:float(np.sqrt(len(y)/(4*counts[c]))) for c in range(4)}\n    # libsvm fits its sigmoid calibrators using only the fit partition.\n    # Outer grouped CV keeps every held-out review group outside that process.\n    return SVC(C=1.,kernel=\'precomputed\',class_weight=cw,probability=True,\n               random_state=SEED,cache_size=512)\n\n\ndef neighbour_class_balance(probabilities,counts,power=.5):\n    """Change neighbour voting\'s class-count correction without refitting IDF."""\n    factor=(np.asarray(counts,dtype=float)/max(counts))**(.5-power)\n    p=probabilities*factor\n    return p/p.sum(axis=1,keepdims=True)\n\n\ndef candidates(base,hard,prob,near,similarity):\n    yield \'v3_reference\',base,{\'kind\':\'reference\',\'alpha\':0.}\n    for alpha in [.05,.1,.2]:\n        yield f\'hard_kernel_{alpha}\',(1-alpha)*base+alpha*np.eye(4)[hard],{\'kind\':\'hard\',\'alpha\':alpha}\n    for alpha in [.15,.3,.5]:\n        yield f\'probability_kernel_{alpha}\',(1-alpha)*base+alpha*prob,{\'kind\':\'probability\',\'alpha\':alpha}\n    for threshold in [.7,.85]:\n        for alpha in [.3,.5]:\n            gate=(similarity>=threshold).astype(float)[:,None]*alpha\n            yield f\'neighbours_{threshold}_{alpha}\',(1-gate)*base+gate*near,{\'kind\':\'neighbours\',\'alpha\':alpha,\'threshold\':threshold}\n\n\ndef select(base,hard,prob,near,similarity,y,folds):\n    rows=[]\n    configurations=list(candidates(base,hard,prob,near,similarity))\n    # The original inverse-square-root class correction is compared with\n    # milder/no correction, using only fit-fold label counts.\n    for power in [0.,.25]:\n        adjusted=np.empty_like(near)\n        for fold in np.unique(folds):\n            mask=folds==fold;counts=np.bincount(y[~mask],minlength=4)\n            adjusted[mask]=neighbour_class_balance(near[mask],counts,power)\n        gate=.3*(similarity>=.85).astype(float)[:,None]\n        configurations.append((f\'neighbours_0.85_0.3_power_{power}\',(1-gate)*base+gate*adjusted,\n            {\'kind\':\'neighbours\',\'alpha\':.3,\'threshold\':.85,\'class_power\':power}))\n    for name,p,recipe in configurations:\n        row=cross_weight(p,y,folds)\n        raw_folds=[macro(y[folds==f],p[folds==f].argmax(1)) for f in np.unique(folds)]\n        row[\'fold_macro_f1\']=[v+d for v,d in zip(raw_folds,row[\'fold_deltas\'])] if row[\'use_weights\'] else raw_folds\n        row.update(name=name,recipe=recipe)\n        rows.append(row)\n    rows.sort(key=lambda r:r[\'selection_score\'],reverse=True)\n    baseline=next(r for r in rows if r[\'recipe\'][\'kind\']==\'reference\')\n    for row in rows:\n        row[\'fold_delta_vs_v5\']=(np.array(row[\'fold_macro_f1\'])-np.array(baseline[\'fold_macro_f1\'])).tolist()\n    return rows\n\n\ndef choose_stable(rows):\n    """Prefer consistent fold gains when aggregate CV differs by < .001."""\n    baseline=next(r for r in rows if r[\'recipe\'][\'kind\']==\'reference\')\n    best=rows[0]\n    if best[\'selection_score\']<baseline[\'selection_score\']+.002:return baseline\n    stable=[r for r in rows if r[\'recipe\'][\'kind\']!=\'reference\'\n            and r[\'selection_score\']>=best[\'selection_score\']-.001\n            and min(r[\'fold_delta_vs_v5\'])>=0]\n    return max(stable,key=lambda r:r[\'selection_score\']) if stable else best\n\n\ndef main(data_dir=\'/content/ds102_data\',output_dir=\'/content/ds102_v6_experiment\',\n         reference_path=\'/content/colab_reference.npz\'):\n    hosted_guard()\n    out=Path(output_dir);out.mkdir(parents=True,exist_ok=True)\n    def status(stage,**values):\n        (out/\'status.json\').write_text(json.dumps(dict(stage=stage,time=time.time(),**values),indent=2))\n    try:\n        df=prepare(load_data(Path(data_dir))[\'train\']);y=df.spam_label.to_numpy()\n        ref=np.load(reference_path);assert np.array_equal(y,ref[\'y\'])\n        hard=np.full(len(y),-1,dtype=int);prob=np.zeros((len(y),4),dtype=np.float32)\n        near=np.zeros_like(prob);similarity=np.zeros(len(y),dtype=np.float32)\n        folds=np.full(len(y),-1);results=[]\n        split=StratifiedGroupKFold(n_splits=3,shuffle=True,random_state=SEED)\n        for fold,(tr,va) in enumerate(split.split(df,y,groups=df.comment)):\n            folds[va]=fold;status(\'features\',fold=fold+1,results=results)\n            f=FeaturesV2();xt,mt=f.fit_transform(df.iloc[tr])\n            xv,mv=f.transform(df.iloc[va].drop(columns=\'spam_label\'))\n            xt=normalize(xt);xv=normalize(xv)\n            mt.data=np.clip(mt.data,-.75,.75);mv.data=np.clip(mv.data,-.75,.75)\n            status(\'neighbours\',fold=fold+1,results=results)\n            near[va],similarity[va]=neighbours(xv,xt,y[tr])\n            status(\'kernel_matrix\',fold=fold+1,results=results)\n            kt=kernel(xt,mt,xt,mt);kv=kernel(xv,mv,xt,mt)\n            status(\'kernel_probability_fit\',fold=fold+1,results=results)\n            m=make_kernel_model(y[tr]);m.fit(kt,y[tr])\n            assert np.array_equal(m.classes_,np.arange(4))\n            hard[va]=m.predict(kv);prob[va]=m.predict_proba(kv)\n            results.append({\'kernel_hard\':macro(y[va],hard[va]),\n                            \'kernel_prob\':macro(y[va],prob[va].argmax(1)),\n                            \'neighbours\':macro(y[va],near[va].argmax(1))})\n            del m,kt,kv,xt,xv,mt,mv,f;gc.collect()\n            np.savez_compressed(out/\'partial.npz\',hard=hard,prob=prob,near=near,\n                                similarity=similarity,y=y,folds=folds)\n        assert np.array_equal(folds,ref[\'folds\'])\n        np.savez_compressed(out/\'oof.npz\',hard=hard,prob=prob,near=near,\n                            similarity=similarity,y=y,folds=folds)\n        rows=select(ref[\'oof\'],hard,prob,near,similarity,y,folds)\n        (out/\'selection.json\').write_text(json.dumps(rows,indent=2))\n        report={\'fold_scores\':results,\'best\':rows[0],\'chosen\':choose_stable(rows),\n            \'selection_rule\':\'Require >=.002 CV gain; within .001 of best, prefer a candidate improving every fold over v5.\',\n            \'reference_v5\':next(r for r in rows if r[\'recipe\'][\'kind\']==\'reference\'),\n            \'pretrained\':False,\'fit_data\':\'train only; dev not read for selection\',\n            \'execution\':\'hosted Colab CPU\',\n            \'caveat\':\'Repeated OOF model selection; weight calibration on existing OOF is not fully nested CV.\'}\n        (out/\'cv_results.json\').write_text(json.dumps(report,indent=2));status(\'complete\',**report)\n        return report\n    except BaseException as exc:\n        status(\'error\',error=str(exc),traceback=traceback.format_exc());raise\n', 'selection_v5.py': '"""Coarse probability reweighting, validated on separate OOF folds."""\nfrom itertools import product\nimport numpy as np\nfrom select_v2 import macro\n\n\ndef fit_weights(probabilities, y):\n    choices = []\n    # A small fixed grid: minority-class recall matters for macro F1.\n    # Identity is included. Never access development or test labels here.\n    for a,b,c in product([.8,1.,1.2,1.4], [.8,1.,1.2,1.4], [.9,1.,1.1]):\n        w=np.array([1.,a,b,c])\n        score=macro(y,(probabilities*w).argmax(1))\n        penalty=.002*float(np.square(np.log(w)).sum())\n        choices.append((score-penalty,-float(np.square(np.log(w)).sum()),w))\n    return max(choices,key=lambda row:row[:2])[2]\n\n\ndef cross_weight(probabilities,y,folds):\n    pred=np.empty(len(y),dtype=int); weights=[]; fold_deltas=[]\n    for fold in np.unique(folds):\n        fit=folds!=fold\n        w=fit_weights(probabilities[fit],y[fit]); weights.append(w.tolist())\n        pred[~fit]=(probabilities[~fit]*w).argmax(1)\n        fold_deltas.append(macro(y[~fit],pred[~fit])-\n            macro(y[~fit],probabilities[~fit].argmax(1)))\n    raw=macro(y,probabilities.argmax(1)); cross=macro(y,pred)\n    use=bool(cross>=raw+.002 and min(fold_deltas)>=-.002)\n    return {\'raw_macro_f1\':raw,\'cross_weight_macro_f1\':cross,\n        \'fold_deltas\':fold_deltas,\'fold_weights\':weights,\'use_weights\':use,\n        \'weights\':fit_weights(probabilities,y).tolist() if use else [1.,1.,1.,1.],\n        \'selection_score\':cross if use else raw}\n\n\ndef compare(reference,new_oof,y,folds,configs):\n    rows=[]\n    row=cross_weight(reference,y,folds)\n    row.update(model_index=-1,config=\'v3_reference\',new_model_weight=0.)\n    rows.append(row)\n    for i,values in enumerate(new_oof):\n        for alpha in [.15,.3,.5,1.]:\n            p=(1-alpha)*reference+alpha*values\n            row=cross_weight(p,y,folds)\n            row.update(model_index=i,config=configs[i],new_model_weight=alpha)\n            rows.append(row)\n    return sorted(rows,key=lambda row:row[\'selection_score\'],reverse=True)\n'}
for name,source in sources.items():
    (root/name).write_text(source)


In [ ]:
import os,sys,subprocess
env=dict(os.environ,OMP_NUM_THREADS='2',OPENBLAS_NUM_THREADS='2',MKL_NUM_THREADS='2')
subprocess.run([sys.executable,'-u','/content/ds102_v7_bundle/experiment_v10.py'],env=env,check=True)
